# Week 11 — Tuesday Lab Notebook  
## Topic: Backend with Flask / FastAPI

This lab is for **Week 11 Tuesday**.  
Today we will build backend thinking in **very simple wording**.

We will cover:

- what backend means
- routing
- URL path parameters
- query parameters
- request body
- JSON parsing
- validation concept
- basic error handling
- API project structure
- small backend examples in **Flask**
- light exposure to **FastAPI**
- testing endpoints without running a public server


## 3-Hour Structure

**Hour 1**
- understand what a backend does
- understand route, endpoint, parameter, and request body
- learn routing in Flask
- learn path parameters and query parameters

**Hour 2**
- build API routes using Flask
- accept JSON body data
- validate required fields
- return proper status codes
- handle basic errors

**Hour 3**
- understand clean backend project structure
- get light introduction to FastAPI
- compare Flask and FastAPI
- solve mini practice and discuss common mistakes


## Setup Cell

Run the next cell once.

It will import required packages.  
If something is missing, it will install it automatically.


In [ ]:
import sys
import subprocess

def ensure_install(package_name):
    try:
        __import__(package_name)
        print(f"{package_name} is already installed.")
    except ImportError:
        print(f"{package_name} not found. Installing...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package_name, "-q"])
        print(f"{package_name} installed successfully.")

# Flask
try:
    from flask import Flask, jsonify, request
    print("Flask is already installed.")
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "flask", "-q"])
    from flask import Flask, jsonify, request
    print("Flask installed successfully.")

# FastAPI + TestClient
try:
    from fastapi import FastAPI, Query
    from fastapi.testclient import TestClient
    print("FastAPI is already installed.")
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "fastapi", "uvicorn", "httpx", "-q"])
    from fastapi import FastAPI, Query
    from fastapi.testclient import TestClient
    print("FastAPI installed successfully.")


## Part A — What is Backend?

Very simple meaning:

- frontend is what user sees
- backend is the part that works behind the scenes

Backend usually handles:

- business logic
- database work
- validation
- authentication
- API responses
- error handling

Example:

A student fills an online registration form.

- **Frontend** shows the form
- **Backend** receives the data
- checks if data is valid
- saves it
- returns a response like `Registration successful`


## Part B — Why Backend Matters

Without backend, many apps cannot work properly.

Examples:

- login system
- attendance system
- e-commerce order system
- result portal
- hospital record system
- weather API
- mobile app data sync

So today we are learning the part that makes apps **useful and dynamic**.


## Part C — Route and Endpoint

A **route** is a path in the backend code.

Examples:

- `/`
- `/students`
- `/students/1`
- `/products`
- `/weather`

An **endpoint** usually means:
a route + an HTTP method

Examples:

- `GET /students`
- `POST /students`
- `DELETE /students/4`

These are different endpoints.


## Part D — Backend Terms to Remember

### 1. Path parameter
A value inside the URL path.

Example:
`/students/5`

Here `5` is a path parameter.

### 2. Query parameter
A value after `?` in URL.

Example:
`/students?city=Lahore&semester=5`

### 3. Body
Extra data sent with the request, often in JSON format.

Example:

```json
{
  "name": "Ali",
  "age": 21
}
```

### 4. Validation
Checking if the data is correct before using or saving it.


## Part E — Flask First Look

Flask is simple and very beginner-friendly.

Basic ideas:

- create app
- define route
- return response

We will not run a public server in today's lab.
We will use Flask `test_client()` so students can test routes inside notebook.


In [ ]:
from flask import Flask, jsonify, request

app = Flask(__name__)

@app.route("/")
def home():
    return "Backend is working!"

print("Simple Flask app created.")


### Teaching point

Explain this clearly:

- `Flask(__name__)` creates the app
- `@app.route("/")` means this function handles the home route
- function returns response


## Part F — Testing a Flask Route in Notebook


In [ ]:
with app.test_client() as client:
    response = client.get("/")
    print("Status code:", response.status_code)
    print("Response text:", response.get_data(as_text=True))


## Part G — Routing Example with Multiple Endpoints


In [ ]:
app2 = Flask(__name__)

@app2.route("/")
def home2():
    return jsonify({"message": "Welcome to our API"})

@app2.route("/about")
def about2():
    return jsonify({"topic": "Backend with Flask"})

@app2.route("/hello/<name>")
def hello2(name):
    return jsonify({"message": f"Hello, {name}!"})

with app2.test_client() as client:
    print(client.get("/").json)
    print(client.get("/about").json)
    print(client.get("/hello/Maria").json)


## Part H — Path Parameters

Path parameters help us get a value directly from the URL.

Example:

- `/students/101`
- `/products/7`

This is useful for:
- getting one item
- updating one item
- deleting one item


In [ ]:
app3 = Flask(__name__)

students_data = {
    1: {"name": "Ali", "semester": 3},
    2: {"name": "Ayesha", "semester": 5},
    3: {"name": "Usman", "semester": 2}
}

@app3.route("/students/<int:student_id>")
def get_student(student_id):
    student = students_data.get(student_id)
    if student is None:
        return jsonify({"error": "Student not found"}), 404
    return jsonify({"student_id": student_id, "data": student})

with app3.test_client() as client:
    print("Existing:", client.get("/students/2").json)
    print("Missing:", client.get("/students/99").status_code, client.get("/students/99").json)


### Teaching point

This route shows two important things:

- how to receive `student_id`
- how to return a custom error if data is missing


## Part I — Query Parameters

Query parameters are useful for filtering and searching.

Example:
- `/search?name=ali`
- `/students?semester=5`
- `/products?category=books&limit=10`


In [ ]:
app4 = Flask(__name__)

students_list = [
    {"name": "Ali", "city": "Lahore", "semester": 3},
    {"name": "Ayesha", "city": "Hafizabad", "semester": 5},
    {"name": "Usman", "city": "Lahore", "semester": 2},
    {"name": "Fatima", "city": "Gujranwala", "semester": 5}
]

@app4.route("/filter_students")
def filter_students():
    city = request.args.get("city")
    semester = request.args.get("semester")

    result = students_list[:]

    if city:
        result = [s for s in result if s["city"].lower() == city.lower()]

    if semester:
        semester = int(semester)
        result = [s for s in result if s["semester"] == semester]

    return jsonify(result)

with app4.test_client() as client:
    print("City filter:", client.get("/filter_students?city=Lahore").json)
    print("Semester filter:", client.get("/filter_students?semester=5").json)
    print("Both filters:", client.get("/filter_students?city=Lahore&semester=3").json)


## Part J — Request Body and JSON Parsing

When client sends data to backend, it often sends JSON body.

In Flask, we usually read it using:

```python
request.get_json()
```

This gives us the JSON data as a Python dictionary.


In [ ]:
app5 = Flask(__name__)

@app5.route("/echo", methods=["POST"])
def echo():
    data = request.get_json()
    return jsonify({
        "received_data": data,
        "message": "Data received successfully"
    }), 201

with app5.test_client() as client:
    response = client.post("/echo", json={"name": "Sara", "course": "Python"})
    print("Status code:", response.status_code)
    print("JSON response:", response.json)


## Part K — Create Endpoint Example

Now we will create a small API that stores books in memory.

Important note:

This is only for learning.
In real projects, data is usually stored in a database.


In [ ]:
app6 = Flask(__name__)

books = [
    {"id": 1, "title": "Python Basics", "author": "Ali"},
    {"id": 2, "title": "Data Science Intro", "author": "Ayesha"}
]

@app6.route("/books", methods=["GET"])
def get_books():
    return jsonify(books)

@app6.route("/books", methods=["POST"])
def add_book():
    data = request.get_json()

    new_book = {
        "id": len(books) + 1,
        "title": data.get("title"),
        "author": data.get("author")
    }
    books.append(new_book)
    return jsonify(new_book), 201

with app6.test_client() as client:
    print("Before add:", client.get("/books").json)
    created = client.post("/books", json={"title": "Machine Learning", "author": "Usman"})
    print("Created:", created.status_code, created.json)
    print("After add:", client.get("/books").json)


## Part L — Why Validation is Needed

Suppose user sends this:

```json
{
  "title": ""
}
```

Problems:
- author missing
- title empty

If backend does not validate, bad data enters the system.

Validation helps us:
- check required fields
- check data type
- check allowed values
- stop bad data early


In [ ]:
app7 = Flask(__name__)

book_store = []

@app7.route("/books", methods=["POST"])
def create_book_validated():
    data = request.get_json()

    if not data:
        return jsonify({"error": "JSON body is required"}), 400

    title = data.get("title")
    author = data.get("author")

    if not title:
        return jsonify({"error": "title is required"}), 400

    if not author:
        return jsonify({"error": "author is required"}), 400

    new_book = {
        "id": len(book_store) + 1,
        "title": title,
        "author": author
    }
    book_store.append(new_book)
    return jsonify(new_book), 201

with app7.test_client() as client:
    good = client.post("/books", json={"title": "Deep Learning", "author": "Fatima"})
    bad1 = client.post("/books", json={"title": "Only Title"})
    bad2 = client.post("/books", json={})

    print("Good request:", good.status_code, good.json)
    print("Bad request 1:", bad1.status_code, bad1.json)
    print("Bad request 2:", bad2.status_code, bad2.json)


### Teaching point

Tell students:

Validation does not mean only checking empty fields.
It can also mean:

- age must be integer
- email must contain `@`
- marks must be between 0 and 100
- price cannot be negative


## Part M — Simple Type Validation Example


In [ ]:
app8 = Flask(__name__)

@app8.route("/register", methods=["POST"])
def register():
    data = request.get_json()

    name = data.get("name")
    age = data.get("age")

    if not name:
        return jsonify({"error": "name is required"}), 400

    if age is None:
        return jsonify({"error": "age is required"}), 400

    if not isinstance(age, int):
        return jsonify({"error": "age must be an integer"}), 400

    if age < 0:
        return jsonify({"error": "age cannot be negative"}), 400

    return jsonify({"message": "Registration successful", "data": data}), 201

with app8.test_client() as client:
    print(client.post("/register", json={"name": "Ali", "age": 20}).json)
    print(client.post("/register", json={"name": "Ali", "age": "20"}).status_code,
          client.post("/register", json={"name": "Ali", "age": "20"}).json)


## Part N — Error Handling Concept

Good backend should not fail silently.

We should return:
- meaningful message
- correct status code

Examples:
- `400` for bad input
- `404` for missing route or item
- `500` for internal problem


In [ ]:
app9 = Flask(__name__)

@app9.route("/divide")
def divide():
    try:
        a = float(request.args.get("a"))
        b = float(request.args.get("b"))
        result = a / b
        return jsonify({"result": result})
    except TypeError:
        return jsonify({"error": "Please provide a and b"}), 400
    except ValueError:
        return jsonify({"error": "a and b must be numbers"}), 400
    except ZeroDivisionError:
        return jsonify({"error": "Division by zero is not allowed"}), 400
    except Exception as e:
        return jsonify({"error": f"Unexpected error: {str(e)}"}), 500

with app9.test_client() as client:
    print("Valid:", client.get("/divide?a=10&b=2").json)
    print("Zero division:", client.get("/divide?a=10&b=0").status_code, client.get("/divide?a=10&b=0").json)
    print("Wrong type:", client.get("/divide?a=ten&b=2").status_code, client.get("/divide?a=ten&b=2").json)


## Part O — Common Backend Mistakes Beginners Make

- forgetting `methods=["POST"]`
- not checking missing JSON body
- assuming every field exists
- returning plain text when JSON is expected
- using wrong status codes
- mixing route logic and data logic carelessly
- not handling bad inputs
- writing all code in one file without structure


## Part P — CRUD Backend Example in Flask

CRUD means:

- **Create**
- **Read**
- **Update**
- **Delete**

We will build a tiny student API.


In [ ]:
app10 = Flask(__name__)

student_db = [
    {"id": 1, "name": "Ali", "semester": 3},
    {"id": 2, "name": "Ayesha", "semester": 5}
]

@app10.route("/students", methods=["GET"])
def read_students():
    return jsonify(student_db)

@app10.route("/students", methods=["POST"])
def create_student():
    data = request.get_json()
    name = data.get("name")
    semester = data.get("semester")

    if not name or semester is None:
        return jsonify({"error": "name and semester are required"}), 400

    new_student = {
        "id": len(student_db) + 1,
        "name": name,
        "semester": semester
    }
    student_db.append(new_student)
    return jsonify(new_student), 201

@app10.route("/students/<int:student_id>", methods=["PUT"])
def update_student(student_id):
    data = request.get_json()

    for student in student_db:
        if student["id"] == student_id:
            student["name"] = data.get("name", student["name"])
            student["semester"] = data.get("semester", student["semester"])
            return jsonify(student)

    return jsonify({"error": "Student not found"}), 404

@app10.route("/students/<int:student_id>", methods=["DELETE"])
def delete_student(student_id):
    for i, student in enumerate(student_db):
        if student["id"] == student_id:
            removed = student_db.pop(i)
            return jsonify({"deleted": removed})

    return jsonify({"error": "Student not found"}), 404

with app10.test_client() as client:
    print("All students:", client.get("/students").json)
    print("Create:", client.post("/students", json={"name": "Usman", "semester": 2}).json)
    print("Update:", client.put("/students/1", json={"semester": 4}).json)
    print("Delete:", client.delete("/students/2").json)
    print("Final data:", client.get("/students").json)


## Part Q — Project Structure Concept

Small beginner programs can be in one file.

But real backend projects should be organized.

Example project structure:

```text
project_folder/
│
├── app.py
├── routes/
│   ├── student_routes.py
│   └── book_routes.py
├── models/
│   └── student.py
├── services/
│   └── student_service.py
├── database/
│   └── db.py
├── schemas/
│   └── validators.py
├── requirements.txt
└── README.md
```

Why this is better:
- code is easier to read
- easier to fix bugs
- easier to add new features
- team members can work separately


## Part R — Simple Meaning of Common Backend Files

### `app.py`
main starting point

### `routes`
contains endpoints

### `models`
represents data structure

### `services`
contains business logic

### `database`
database connection work

### `schemas` or `validators`
validation rules

Students do not need full architecture today.
They just need the **idea of separation**.


## Part S — Flask vs FastAPI

### Flask
- very simple
- very easy for beginners
- flexible
- less automatic validation by default

### FastAPI
- modern API framework
- automatic docs
- fast development
- strong validation support
- very popular for API-focused projects

For teaching beginners:
- Flask is easier first
- FastAPI is excellent after basics


## Part T — First FastAPI Example

Now we will see a very small FastAPI example.

We will use `TestClient` so the notebook can test routes directly.


In [ ]:
from fastapi import FastAPI
from fastapi.testclient import TestClient

fast_app = FastAPI()

@fast_app.get("/")
def home_fast():
    return {"message": "FastAPI is working"}

client = TestClient(fast_app)
response = client.get("/")
print("Status code:", response.status_code)
print("JSON response:", response.json())


### Teaching point

Students should notice this style:

- `@fast_app.get("/")`
- function returns dictionary
- FastAPI automatically turns it into JSON response


## Part U — FastAPI Path and Query Parameters


In [ ]:
from fastapi import Query

fast_app2 = FastAPI()

@fast_app2.get("/items/{item_id}")
def get_item(item_id: int, discount: float = Query(default=0)):
    return {
        "item_id": item_id,
        "discount": discount
    }

client2 = TestClient(fast_app2)

print(client2.get("/items/10").json())
print(client2.get("/items/10?discount=15.5").json())


## Part V — Validation Concept in FastAPI

One nice thing in FastAPI:
you can add type hints directly.

Example:

- `item_id: int`
- `discount: float`

If user sends wrong type, FastAPI can reject it automatically.

This is one reason many developers like FastAPI.


## Part W — FastAPI Request Body with Validation


In [ ]:
from pydantic import BaseModel

fast_app3 = FastAPI()

class UserData(BaseModel):
    name: str
    age: int

@fast_app3.post("/users")
def create_user(user: UserData):
    return {
        "message": "User created successfully",
        "data": user.dict()
    }

client3 = TestClient(fast_app3)

good = client3.post("/users", json={"name": "Sara", "age": 22})
bad = client3.post("/users", json={"name": "Sara", "age": "twenty two"})

print("Good request:", good.status_code, good.json())
print("Bad request:", bad.status_code, bad.json())


## Part X — Why FastAPI Validation Feels Powerful

In Flask:
- you often write manual checks yourself

In FastAPI:
- type hints and models help a lot
- invalid data is caught early
- automatic documentation can also be created

But for students, it is still important to understand **manual validation logic** first.


## Part Y — Mini Comparison Table

| Feature | Flask | FastAPI |
|---|---|---|
| Beginner friendliness | Very high | Medium |
| API speed of development | Good | Very good |
| Validation | Manual mostly | Strong built-in style |
| Automatic docs | No default | Yes |
| Best use in this course | First backend learning | Next step after basics |


## Part Z — A Small Practice Scenario

Suppose you are building an institute management system.

Possible endpoints:

- `GET /students`
- `POST /students`
- `GET /teachers`
- `POST /courses`
- `PUT /students/5`
- `DELETE /students/5`

Ask students:
1. which route reads all students?
2. which route adds a course?
3. where do we send updated semester?
4. what status code is suitable after creating data?


In [ ]:
practice_answers = {
    1: "GET /students",
    2: "POST /courses",
    3: "Usually in request body of PUT /students/5",
    4: "201 Created"
}

practice_answers


## In-Class Mini Practice

Try these one by one:

1. Create a Flask route `/teacher/<name>` that returns welcome message.
2. Create a Flask route `/square/<int:num>` that returns square of number.
3. Create a route `/search_course` that reads query parameter `title`.
4. Create a POST route `/login` that accepts JSON with username and password.
5. Add validation that password must not be empty.


## Common Interview / Viva Questions from Today

- What is backend?
- What is difference between route and endpoint?
- What is path parameter?
- What is query parameter?
- What is request body?
- Why is validation important?
- Why do we use status codes?
- What is difference between Flask and FastAPI?
- What is CRUD?
- Why do real projects need folder structure?


## 10 Lab Tasks

Write code for the following tasks.

### Task 1
Create a Flask route `/hello/<name>` that returns JSON welcome message.

### Task 2
Create a Flask route `/multiply` that reads query parameters `a` and `b` and returns their product.

### Task 3
Create a Flask POST route `/student` that accepts JSON with `name` and `semester`.

### Task 4
Add validation in Task 3 so missing `name` gives `400`.

### Task 5
Create a small in-memory list of books and build `GET /books`.

### Task 6
Add `POST /books` to insert a new book into the list.

### Task 7
Create `PUT /books/<id>` to update a book title.

### Task 8
Create `DELETE /books/<id>` to remove a book.

### Task 9
Write a FastAPI route `/product/{product_id}` with one optional query parameter `discount`.

### Task 10
Create a FastAPI POST route with a Pydantic model for user registration having:
- `name: str`
- `email: str`
- `age: int`


## Optional Homework Challenge

Build a mini **course management API**.

Requirements:

- use Flask **or** FastAPI
- create routes for:
  - add course
  - view all courses
  - update course fee
  - delete course
- validate that:
  - course name is required
  - fee cannot be negative
- return proper status codes
- keep code clean and organized

Extra challenge:
separate route logic and data logic into different sections or files.


## Summary of Today

Today we learned:

- backend basics
- routing
- path and query parameters
- JSON request body
- validation concept
- error handling
- CRUD endpoints
- clean project structure idea
- Flask basics
- FastAPI basics

This is enough foundation for the next class where database and CRUD can become more practical.
